# Shared Report Header

Reusable metadata presentation for historical reports.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import os
from IPython.display import display, Markdown
import pandas as pd

def get_report_metadata():
    """Return common report and database metadata.

    :param: This method accepts no parameters.
    :return: A dictionary containing database and observation-session metadata.
    """
    # Resolve the configured tracker database and load session boundaries from SQL.
    database_path = Path(os.environ[DB_PATH_VARIABLES['tracker']]).expanduser().absolute()
    query = construct_query('tracker', 'reports', 'report-metadata.sql', {})
    session_metadata = query_data('tracker', query).iloc[0]

    # Keep filesystem and generation timestamps timezone-aware and readable.
    modified_at = datetime.fromtimestamp(database_path.stat().st_mtime, timezone.utc)
    return {
        'Report generated': datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC'),
        'Database': str(database_path),
        'Database modified': modified_at.strftime('%Y-%m-%d %H:%M:%S UTC'),
        'Earliest session': session_metadata['Earliest Session'],
        'Latest session': session_metadata['Latest Session'],
        'Sessions included': int(session_metadata['Session Count']),
    }

def display_report_header(report_title):
    """Display a consistent report title and metadata table.

    :param report_title: Human-readable title for the report.
    :return: The metadata DataFrame displayed beneath the title.
    """
    # Present metadata vertically so long paths and timestamps remain readable.
    metadata = get_report_metadata()
    metadata_frame = pd.DataFrame(metadata.items(), columns=['Report metadata', 'Value'])
    display(Markdown(f'# {report_title}'))
    display(metadata_frame.style.hide(axis='index'))
    return metadata_frame
